# Export MDS coordinates
Computes 2D and 3D MDS coordinates for each sound category and saves them to `data/mds/mds_2d.csv` and `data/mds/mds_3d.csv`.

In [1]:
# Install dependencies
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn import manifold

In [11]:
# Configuration

INPUT_FILE = "data/similarity_data.csv"
OUTPUT_DIR = "data"
MDS_SEED   = 123

In [13]:
# Load & aggregate

os.makedirs(OUTPUT_DIR, exist_ok=True)

similarity_data = pd.read_csv(INPUT_FILE)

similarity_data = (
    similarity_data
    .groupby(["category", "sound1", "sound2"], as_index=False)
    .agg(response=("response", "mean"))
)

categories = sorted(similarity_data["category"].unique())
print(f"Found {len(categories)} categories: {categories}")

Found 53 categories: ['BirdSong', 'BirdSquawk', 'BirdsFlyingOff', 'CarHorn', 'CarSkidding', 'CarStarting', 'CatHissing', 'CatMeowing', 'CatPurring', 'ChildrenPlaying', 'CocktailParty', 'CoughingAdultFemale', 'CoughingAdultMale', 'CowMoo', 'CrowdApplause', 'CrowdCheering', 'CryingAdultFemale', 'CryingAdultMale', 'CryingChild', 'DogBarking', 'DogGrowling', 'DogHowling', 'Drilling', 'DrillingPneumatic', 'Fire', 'Footsteps', 'Hammer', 'Helicopter', 'HorseGalloping', 'HorseNeighing', 'HorseSnorting', 'InsectsBuzzing', 'InsectsStridulation', 'JetPassing', 'LaughingAdultFemale', 'LaughingAdultMale', 'LaughingChild', 'LaughingGroup', 'Propeller', 'Rain', 'River', 'Sawing', 'SawingManual', 'SeaWaves', 'Sneezing', 'Snoring', 'ThroatClearing', 'Thunder', 'TrainBrakes', 'TrainSignal', 'TrainWagons', 'Waterfall', 'Wind']


In [15]:
# Compute MDS for each category

records_2d = []
records_3d = []

for cat in categories:
    cat_data   = similarity_data[similarity_data["category"] == cat]
    sounds     = list(cat_data["sound1"].unique())   # preserves insertion order
    sound_to_i = {s: i for i, s in enumerate(sounds)}
    n          = len(sounds)

    # Build dissimilarity matrix
    dissim = np.zeros((n, n))
    for _, row in cat_data.iterrows():
        i = sound_to_i[row["sound1"]]
        j = sound_to_i[row["sound2"]]
        d = 1.0 - row["response"]
        dissim[i, j] = d
        dissim[j, i] = d

    # Fit MDS — same settings as the Streamlit app
    coords_2d = manifold.MDS(n_components=2, dissimilarity="precomputed", random_state=MDS_SEED).fit_transform(dissim)
    coords_3d = manifold.MDS(n_components=3, dissimilarity="precomputed", random_state=MDS_SEED).fit_transform(dissim)

    for i, sound in enumerate(sounds):
        records_2d.append({"category": cat, "sound": sound, "x": coords_2d[i, 0], "y": coords_2d[i, 1]})
        records_3d.append({"category": cat, "sound": sound, "x": coords_3d[i, 0], "y": coords_3d[i, 1], "z": coords_3d[i, 2]})

    print(f"  ✓  {cat}  ({n} sounds)")

  ✓  BirdSong  (10 sounds)
  ✓  BirdSquawk  (10 sounds)
  ✓  BirdsFlyingOff  (10 sounds)
  ✓  CarHorn  (10 sounds)
  ✓  CarSkidding  (10 sounds)
  ✓  CarStarting  (10 sounds)
  ✓  CatHissing  (10 sounds)
  ✓  CatMeowing  (10 sounds)
  ✓  CatPurring  (10 sounds)
  ✓  ChildrenPlaying  (10 sounds)
  ✓  CocktailParty  (10 sounds)
  ✓  CoughingAdultFemale  (10 sounds)
  ✓  CoughingAdultMale  (10 sounds)
  ✓  CowMoo  (10 sounds)
  ✓  CrowdApplause  (10 sounds)
  ✓  CrowdCheering  (10 sounds)
  ✓  CryingAdultFemale  (10 sounds)
  ✓  CryingAdultMale  (10 sounds)
  ✓  CryingChild  (10 sounds)
  ✓  DogBarking  (10 sounds)
  ✓  DogGrowling  (10 sounds)
  ✓  DogHowling  (10 sounds)
  ✓  Drilling  (10 sounds)
  ✓  DrillingPneumatic  (10 sounds)
  ✓  Fire  (10 sounds)
  ✓  Footsteps  (10 sounds)
  ✓  Hammer  (10 sounds)
  ✓  Helicopter  (10 sounds)
  ✓  HorseGalloping  (10 sounds)
  ✓  HorseNeighing  (10 sounds)
  ✓  HorseSnorting  (10 sounds)
  ✓  InsectsBuzzing  (10 sounds)
  ✓  InsectsStridulatio

In [17]:
# Save CSVs

path_2d = os.path.join(OUTPUT_DIR, "mds_2d.csv")
path_3d = os.path.join(OUTPUT_DIR, "mds_3d.csv")

pd.DataFrame(records_2d).to_csv(path_2d, index=False)
pd.DataFrame(records_3d).to_csv(path_3d, index=False)

print(f"Wrote {path_2d}")
print(f"Wrote {path_3d}")

Wrote data/mds_2d.csv
Wrote data/mds_3d.csv


In [19]:
# Check

df_2d = pd.read_csv(path_2d)
df_3d = pd.read_csv(path_3d)

print(f"2D: {df_2d.shape[0]} rows, categories: {sorted(df_2d['category'].unique())}")
print(f"3D: {df_3d.shape[0]} rows, categories: {sorted(df_3d['category'].unique())}")
df_2d.head()

2D: 530 rows, categories: ['BirdSong', 'BirdSquawk', 'BirdsFlyingOff', 'CarHorn', 'CarSkidding', 'CarStarting', 'CatHissing', 'CatMeowing', 'CatPurring', 'ChildrenPlaying', 'CocktailParty', 'CoughingAdultFemale', 'CoughingAdultMale', 'CowMoo', 'CrowdApplause', 'CrowdCheering', 'CryingAdultFemale', 'CryingAdultMale', 'CryingChild', 'DogBarking', 'DogGrowling', 'DogHowling', 'Drilling', 'DrillingPneumatic', 'Fire', 'Footsteps', 'Hammer', 'Helicopter', 'HorseGalloping', 'HorseNeighing', 'HorseSnorting', 'InsectsBuzzing', 'InsectsStridulation', 'JetPassing', 'LaughingAdultFemale', 'LaughingAdultMale', 'LaughingChild', 'LaughingGroup', 'Propeller', 'Rain', 'River', 'Sawing', 'SawingManual', 'SeaWaves', 'Sneezing', 'Snoring', 'ThroatClearing', 'Thunder', 'TrainBrakes', 'TrainSignal', 'TrainWagons', 'Waterfall', 'Wind']
3D: 530 rows, categories: ['BirdSong', 'BirdSquawk', 'BirdsFlyingOff', 'CarHorn', 'CarSkidding', 'CarStarting', 'CatHissing', 'CatMeowing', 'CatPurring', 'ChildrenPlaying', 'C

,category,sound,x,y
0,BirdSong,1,0.249077,-1.222107
1,BirdSong,2,-1.115680,-0.733268
2,BirdSong,3,0.600490,-0.360622
3,BirdSong,4,0.101829,2.150905
4,BirdSong,5,-0.146830,0.288148
